Lab 4: LLMs and Prompt Engineering for Decision Support
Duration: 2 weeks [30 Jul - 13 Aug, 2026] Due Date: 13th August, 2026 Format: Jupyter Notebook / Google Colab + external APIs + GitHub version control Grading: This is a graded lab.

Student Name: Louisa-Lois Student ID: 13532028

Part 0: Repository and API-key setup

In [ ]:
!git clone https://github.com/Louisa-Lois/lab-4-llm-decision-support.git
%cd lab-4-llm-decision-support

Cloning into 'lab-4-llm-decision-support'...
remote: Enumerating objects: 23, done.
remote: Counting objects: 100% (23/23), done.
remote: Compressing objects: 100% (18/18), done.
remote: Total 23 (delta 4), reused 21 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (23/23), 25.63 KiB | 25.63 MiB/s, done.
Resolving deltas: 100% (4/4), done.
/content/lab-4-llm-decision-support/lab-4-llm-decision-support/lab-4-llm-decision-support/lab-4-llm-decision-support


In [ ]:
!pip install openai -q

In [ ]:
# API-key setup

import os
from google.colab import userdata

API_KEY = userdata.get("GROQ_API_KEY")

# OpenAI-compatible client
from openai import OpenAI

client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.groq.com/openai/v1",
)
MODEL = "openai/gpt-oss-120b"

print("Client ready.")

Client ready.


Section 1 — Talking to an LLM Programmatically

Part 1.1 — Your first API call

In [ ]:
# Part 1.1: Your first API call

def ask_llm(user_prompt, system_prompt="You are a helpful assistant.",
            temperature=0.7, max_tokens=500):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return response.choices[0].message.content


# Call it once with a simple question and print the answer
answer = ask_llm("What is microfinance, in one sentence?")
print("Answer:")
print(answer)

Answer:
Microfinance is the provision of small‑scale financial services—such as micro‑loans, savings, and insurance—to low‑income individuals or communities who lack access to traditional banking.


In [ ]:
# Print response.usage as well
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user",   "content": "What is microfinance, in one sentence?"},
    ],
    temperature=0.7,
    max_tokens=500,
)

print("\nToken usage:")
print(response.usage)


Token usage:
CompletionUsage(completion_tokens=65, prompt_tokens=89, total_tokens=154, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=22, rejected_prediction_tokens=None), prompt_tokens_details=None, queue_time=0.269056733, prompt_time=0.003426912, completion_time=0.135737415, total_time=0.139164327)


**Student Reasoning - Anatomy of a call**

**1. Difference between system and user roles:**
The system role sets the model's overall behaviour, persona, and rules for the
entire conversation, it's set once and shapes how the model responds to
everything that follows. The user role is the actual question or task being
asked right now. Example: system = "You are a helpful assistant" (sets the
general behaviour), user = "What is microfinance, in one sentence?" (the
specific request). In this lab, the system role will later be used to give
the model a specific job (e.g. "You are an assistant to a microfinance loan
officer...") while the user role delivers the actual letter to process.

**2. What is a token, and why bill per token?**
A token is roughly a piece of a word, sometimes a whole word, sometimes
part of one. My test call used 89 prompt tokens (the question + system
message) and 76 completion tokens (the answer), totaling 165 tokens. API
providers bill per token rather than per request because the actual
computational cost of generating a response scales directly with how much
text is processed and produced, a one-word answer costs far less compute
than a 500-word essay, even though both are "one request." Billing per
token ties cost directly to actual resource usage.

Part 1.2 — Temperature: the randomness dial

In [ ]:
# Part 1.2: Temperature — the randomness dial

question = "Suggest a name for a savings product for market traders in Accra."

# 5 calls at temperature = 0.0
print("=== Temperature = 0.0 ===\n")
answers_temp0 = []
for i in range(5):
    answer = ask_llm(question, temperature=0.0)
    answers_temp0.append(answer)
    print(f"Run {i+1}: {answer}\n")

# calls at temperature = 1.2
print("\n=== Temperature = 1.2 ===\n")
answers_temp12 = []
for i in range(5):
    answer = ask_llm(question, temperature=1.2)
    answers_temp12.append(answer)
    print(f"Run {i+1}: {answer}\n")

=== Temperature = 0.0 ===

Run 1: **Name Ideas for a Savings Product Targeted at Market Traders in Accra**

| # | Suggested Name | Why It Works (Brief Rationale) |
|---|----------------|--------------------------------|
| 1 | **“Kokro‑Kokro Savings”** | *Kokro* means “tomorrow” in Twi – conveys the idea of saving today for a better tomorrow, a message that resonates with traders who plan for the next market day. |
| 2 | **“Bɔkɔɔ Vault”** | *Bɔkɔɔ* translates to “prosperity” or “goodness.” Pairing it with “Vault” adds a sense of security and modern banking. |
| 3 | **“MarketMate Fund”** | Directly references the market environment, positioning the product as a trusted “mate” that helps traders grow their cash flow. |
| 4 | **“Sika Sika Saver”** | *Sika* means “money” in Twi; the repetition emphasizes “more money” and is easy to remember. |
| 5 | **“Adwuma Nest”** | *Adwuma* means “work” or “business.” A “nest” evokes a safe place to store earnings for future growth. |
| 6 | **“Kente Cas

**Student Reasoning - Temperature**

At temperature 0.0, the answers were mostly repeated, Runs 1, 3, and 5
gave nearly identical lists (same names: Kokro-Kokro Savings, Bɔkɔɔ Bank,
Kente Kash...), and Runs 2 an                                                                                                                                                            d 4 were identical to each other but
different from 1/3/5. This shows temp=0.0 makes the model pick the most
likely response almost every time, though it's not perfectly deterministic.

At temperature 1.2, every single run gave a completely different set of
names and even different formats (some had taglines, one focused on a
single name instead of a list). This shows higher temperature makes the
model much more random and creative.

For the loan decision-support system, temperature=0.0 is the right
choice. The system needs to extract facts and give consistent advice
from a loan letter, not be creative. A loan officer needs the same
letter to produce the same summary and risk assessment every time, not
a different answer depending on luck.

Section 2 — The Dataset: Loan Application Letters

In [ ]:
LETTERS = {
"L001": """Dear Sir/Madam,
My name is Akosua Mensah and I have been selling provisions at Makola Market for 12 years.
I am applying for a loan of GHS 8,000 to buy a deep freezer and expand into frozen foods.
My current stall makes about GHS 900 profit each month. I have saved GHS 2,500 with your
susu scheme over the past two years and I have never missed a contribution. I can repay
GHS 450 monthly over 20 months. My sister, a teacher, will stand as my guarantor.
Thank you for considering my application.""",

"L002": """Hello,
I am Kwame Boateng, a commercial driver in Kumasi. I need GHS 25,000 urgently to repair my
trotro engine and settle some personal debts. Business has been slow but it will surely
pick up after the festive season. I can pay back whenever the money comes. I do not have
collateral at the moment but God willing everything will be fine. Please help me quickly.""",

"L003": """Dear Loan Committee,
I am Efua Darko, owner of Darko Fashions, a registered dressmaking business in Takoradi
(registration no. BN-2019-4482). I employ three apprentices. I request GHS 15,000 to
purchase two industrial sewing machines and fabric stock ahead of the Christmas season.
Last year my December revenue alone was GHS 22,000; monthly profit averages GHS 2,800.
I hold a fixed deposit of GHS 5,000 with GCB which I can pledge. Proposed repayment:
GHS 1,100 monthly for 15 months. Attached are my sales records for the past 18 months.""",

"L004": """Good day,
My name is Yaw Owusu. I want a loan for my poultry farm at Nsawam. The amount is GHS 12,000
for feed and 500 new layers. I started the farm last year. Sometimes I make good money,
around GHS 1,500 in a good month, but bird flu affected us in March and I lost many birds.
I am rebuilding now. I can repay in 18 months. My uncle has agreed to guarantee the loan
with his taxi.""",

"L005": """Dear Manager,
I am writing on behalf of the Adenta Women's Weaving Cooperative (14 members). We seek
GHS 30,000 to buy a bulk order of yarn directly from the factory, cutting out middlemen and
raising our margins from 15% to about 35%. The cooperative has operated for 6 years and
holds GHS 9,000 in our group account. We propose repayment of GHS 2,000 monthly over
16 months, backed by our group savings and joint liability agreement.""",

"L006": """Hi,
This is Kofi. I saw your advert. I want GHS 50,000 to start a car washing business, a
provision shop, and also import phones from Dubai. I am 22 and full of energy. I have not
started any of these yet but my friends say I am very business minded. I will pay back in
one year when the businesses are booming. No collateral but I am trustworthy.""",
}

# Gold-standard labels for three letters (for Section 4 evaluation):
GOLD = {
  "L001": {"applicant_name": "Akosua Mensah", "amount_ghs": 8000,  "purpose": "buy deep freezer / expand into frozen foods",
           "monthly_profit_ghs": 900,  "has_collateral_or_guarantor": True,  "repayment_months": 20},
  "L003": {"applicant_name": "Efua Darko",    "amount_ghs": 15000, "purpose": "industrial sewing machines and fabric stock",
           "monthly_profit_ghs": 2800, "has_collateral_or_guarantor": True,  "repayment_months": 15},
  "L006": {"applicant_name": "Kofi",          "amount_ghs": 50000, "purpose": "car wash, provision shop, phone imports",
           "monthly_profit_ghs": None, "has_collateral_or_guarantor": False, "repayment_months": 12},
}

print(f"{len(LETTERS)} letters loaded.")

6 letters loaded.


Section 3 — Prompt Engineering for the Decision Support System

Part 3.1 — Component 1: Summarization

In [ ]:
# Part 3.1: Component 1 — Summarization

# V1: naive prompt
SUMMARY_PROMPT_V1 = "Summarize this:"

print("=== SUMMARY V1 — L002 ===\n")
v1_l002 = ask_llm(f"{SUMMARY_PROMPT_V1}\n\n{LETTERS['L002']}", temperature=0)
print(v1_l002)

print("\n\n=== SUMMARY V1 — L006 ===\n")
v1_l006 = ask_llm(f"{SUMMARY_PROMPT_V1}\n\n{LETTERS['L006']}", temperature=0)
print(v1_l006)

=== SUMMARY V1 — L002 ===

Kwame Boateng, a commercial driver in Kumasi, is requesting an urgent loan of GHS 25,000 to repair his trotro engine and cover personal debts. He notes that business has been slow but expects improvement after the festive season, and he can repay the loan once funds become available, though he currently has no collateral. He is asking for quick assistance.


=== SUMMARY V1 — L006 ===

**Summary**

Kofi, a 22‑year‑old entrepreneur, is seeking a GHS 50,000 loan to launch three ventures—a car‑washing service, a provision shop, and an import business for phones from Dubai. He has no collateral but claims to be trustworthy and confident that the businesses will generate enough profit to repay the loan within one year. His friends describe him as business‑minded, though he has not yet started any of the enterprises.


In [ ]:
# V2: engineered prompt with role + constraints

SUMMARY_SYSTEM_V2 = """You are an assistant to a microfinance loan officer in Ghana.
Your job is to summarize loan application letters into short, factual briefs
the officer can scan quickly.

Rules:
- Write exactly 3-4 sentences.
- Be strictly factual and neutral in tone — do not add opinions or judgments.
- Do NOT invent, assume, or infer any detail that is not explicitly stated in the letter.
- If a key detail (amount, purpose, repayment plan) is missing or vague, say so plainly
  rather than filling it in.
"""

def SUMMARY_PROMPT_V2(letter_text):
    return f"Summarize this loan application:\n\n{letter_text}"


print("=== SUMMARY V2 — L002 ===\n")
v2_l002 = ask_llm(
    SUMMARY_PROMPT_V2(LETTERS['L002']),
    system_prompt=SUMMARY_SYSTEM_V2,
    temperature=0
)
print(v2_l002)

print("\n\n=== SUMMARY V2 — L006 ===\n")
v2_l006 = ask_llm(
    SUMMARY_PROMPT_V2(LETTERS['L006']),
    system_prompt=SUMMARY_SYSTEM_V2,
    temperature=0
)
print(v2_l006)

=== SUMMARY V2 — L002 ===

Kwame Boateng, a commercial driver based in Kumasi, is applying for a loan of GHS 25,000. He states the funds are needed to repair his trotro engine and to settle personal debts. He indicates that repayment will be made “whenever the money comes,” but provides no specific repayment schedule or timeline. He also notes that he does not have collateral available at this time.


=== SUMMARY V2 — L006 ===

Kofi requests a loan of GHS 50,000. He intends to use the funds to start a car‑washing business, a provision shop, and to import phones from Dubai. He is 22 years old and says he will repay the loan in one year when the businesses are booming, but provides no detailed repayment schedule. No collateral is offered.


In [ ]:
# Side-by-side comparison

print("=" * 70)
print("V1 vs V2 COMPARISON — L002")
print("=" * 70)
print(f"\nV1:\n{v1_l002}")
print(f"\nV2:\n{v2_l002}")

print("\n\n" + "=" * 70)
print("V1 vs V2 COMPARISON — L006")
print("=" * 70)
print(f"\nV1:\n{v1_l006}")
print(f"\nV2:\n{v2_l006}")

V1 vs V2 COMPARISON — L002

V1:
Kwame Boateng, a commercial driver in Kumasi, is requesting an urgent loan of GHS 25,000 to repair his trotro engine and cover personal debts. He notes that business has been slow but expects improvement after the festive season, and he can repay the loan once funds become available, though he currently has no collateral. He is asking for quick assistance.

V2:
Kwame Boateng, a commercial driver based in Kumasi, is applying for a loan of GHS 25,000. He states the funds are needed to repair his trotro engine and to settle personal debts. He indicates that repayment will be made “whenever the money comes,” but provides no specific repayment schedule or timeline. He also notes that he does not have collateral available at this time.


V1 vs V2 COMPARISON — L006

V1:
**Summary**

Kofi, a 22‑year‑old entrepreneur, is seeking a GHS 50,000 loan to launch three ventures—a car‑washing service, a provision shop, and an import business for phones from Dubai. He has

**Student Reasoning — Summarization prompts**

**1. Problems in V1 that V2 fixed:**

V1 added a "**Summary**" header, inconsistent formatting for something meant
to be scanned quickly. V1 also used more interpretive language: it called
Kofi "a 22-year-old entrepreneur" and said he was "confident that the
businesses will generate enough profit to repay the loan", these are
judgments/characterizations not stated in the letter that way, just his own
claims being reported as fact. V2 fixed this by sticking to neutral phrasing
like "He states..." and "He intends to..."

V1 also didn't clearly flag missing information. For L002, V1 just said
"he promises to repay the loan as soon as funds become available" without
noting that's a red flag. V2 explicitly wrote "without specifying a
repayment schedule or timeline", directly surfacing the gap for the
loan officer instead of just restating the vague promise.

**2. Why "no invented details" matters, and its name in the literature:**

This failure mode is called hallucination, when an LLM states something
confidently that isn't actually supported by the source text. For a loan
officer, a hallucinated detail (like a wrong repayment amount or an
invented collateral claim) could directly lead to a bad lending decision.
Since the summary is meant to replace reading the full letter, any
invented detail becomes invisible to the officer, they have no way to
catch the error without re-reading the original letter themselves,
defeating the entire purpose of summarization.

Part 3.2 — Component 2: Structured extraction (JSON)

In [ ]:
# Part 3.2: Component 2 — Structured extraction (JSON)
import pandas as pd
import json
import re

# The extraction prompt with schema + few-shot example
EXTRACT_SYSTEM_PROMPT = """You are a data extraction assistant for a microfinance loan officer.
You will be given a loan application letter. Extract the following fields and
return ONLY a valid JSON object with EXACTLY these keys — no extra text,
no explanation, no markdown formatting:

- applicant_name (string)
- amount_ghs (number)
- purpose (string)
- monthly_profit_ghs (number or null)
- has_collateral_or_guarantor (boolean)
- repayment_months (number or null)

RULES:
- If a field is not explicitly stated in the letter, use null. Do NOT guess or infer.
- Return ONLY the JSON object. No commentary before or after.

EXAMPLE:

Letter:
"Dear Sir, my name is Ama Serwaa. I run a small chop bar in Tema and need
GHS 4,000 to buy new cooking equipment. My guarantor is my brother, a civil
servant. I did not mention my monthly income in this letter."

Output:
{
  "applicant_name": "Ama Serwaa",
  "amount_ghs": 4000,
  "purpose": "buy new cooking equipment for chop bar",
  "monthly_profit_ghs": null,
  "has_collateral_or_guarantor": true,
  "repayment_months": null
}
"""


def EXTRACT_PROMPT(letter_text):
    return f"Extract the fields from this loan application letter:\n\n{letter_text}"


# Extraction function with graceful failure handling
def extract_fields(letter_text):
    raw_output = ask_llm(
        EXTRACT_PROMPT(letter_text),
        system_prompt=EXTRACT_SYSTEM_PROMPT,
        temperature=0,
        max_tokens=800,
    )

    # Strip json fences if the model added them anyway
    cleaned = re.sub(r"^```json\s*|\s*```$", "", raw_output.strip(), flags=re.MULTILINE)
    cleaned = cleaned.strip()

    try:
        result = json.loads(cleaned)
        return result
    except json.JSONDecodeError:
        print(f"⚠️ Warning: failed to parse JSON. Raw output was:\n{raw_output}\n")
        return None


# Run on all six letters
extraction_results = {}
for letter_id, letter_text in LETTERS.items():
    result = extract_fields(letter_text)
    extraction_results[letter_id] = result

# Collect into a DataFrame
extraction_df = pd.DataFrame(extraction_results).T
extraction_df.index.name = "letter_id"

print("=== Extraction Results ===")
display(extraction_df)

=== Extraction Results ===


,applicant_name,amount_ghs,purpose,monthly_profit_ghs,has_collateral_or_guarantor,repayment_months
letter_id,,,,,,
L001,Akosua Mensah,8000,buy a deep freezer and expand into frozen foods,900,True,20
L002,Kwame Boateng,25000,repair my trotro engine and settle some person...,None,False,None
L003,Efua Darko,15000,purchase two industrial sewing machines and fa...,2800,True,15
L004,Yaw Owusu,12000,feed and new layers for poultry farm,1500,True,18
L005,Adenta Women's Weaving Cooperative,30000,buy a bulk order of yarn,None,True,16
L006,Kofi,50000,"start a car washing business, a provision shop...",None,False,12


**Student Reasoning - Structured extraction**

**1. Why the few-shot example must NOT come from the six letters:**
If the example came from one of the six letters being processed, the model
could pattern-match superficial features of that specific letter (its exact
phrasing, its specific applicant) rather than learning the general task of
extraction. Using a separate fictional example (Ama Serwaa) forces the model
to generalize the extraction pattern rather than memorize an answer it
already effectively "saw."

**2. Why "use null, do not guess" matters:**
Without this instruction, an LLM tends to fill in plausible-sounding values
for missing fields rather than admitting the information isn't there, a
form of hallucination. In this system, the extraction results in L002 and
L006 confirm the instruction worked: monthly_profit_ghs and repayment_months
correctly came back as null/None where the letters genuinely never state
those numbers, rather than the model inventing a plausible profit figure.

**3. Why temperature=0 is right for extraction but not creative tasks:**
Extraction is a task with one objectively correct answer per field, the
number either was stated as GHS 8,000 or it wasn't. There's no room for
"creative variation" in a correct extraction; any variation is either noise
or error. Temperature=0 makes the model deterministic, picking the single
most likely token every time, which maximizes consistency and reliability.
Creative tasks like the savings-product naming in Part 1.2 benefit from
higher temperature because multiple different answers can all be equally
valid or interesting, there's no single "correct" name to converge on.

Part 3.3 — Component 3: The decision-support brief

In [ ]:
# Part 3.3: Component 3 — The decision-support brief

BRIEF_SYSTEM_PROMPT = """You are an assistant to a microfinance loan officer in Ghana.
You support the officer's decision-making — you do NOT make the final decision.

Given a loan application letter and its extracted structured data, produce a brief with
exactly these four sections:

1. Strengths — bullet points, grounded strictly in what the letter actually states.
2. Risks / Red Flags — bullet points identifying concerns for the loan officer to consider.
3. Missing Information — what the officer should request before deciding, if anything.
4. Suggested Next Step — ONE of: "invite for interview", "request documents",
   "flag for senior review", or "proceed to standard review".

RULES:
- NEVER output "approve" or "reject" or any equivalent final decision.
- The final lending decision is always made by a human loan officer, not by you.
- Do not invent details not present in the letter or extracted data.
- Be factual and neutral — this is a decision-support tool, not a decision-maker.
"""


def BRIEF_PROMPT(letter_text, extracted_json):
    return f"""Loan application letter:
{letter_text}

Extracted data:
{json.dumps(extracted_json, indent=2)}

Produce the decision-support brief."""


# Generate briefs for all six letters
briefs = {}
for letter_id, letter_text in LETTERS.items():
    extracted = extraction_results[letter_id]
    brief = ask_llm(
        BRIEF_PROMPT(letter_text, extracted),
        system_prompt=BRIEF_SYSTEM_PROMPT,
        temperature=0,
        max_tokens=600,
    )
    briefs[letter_id] = brief

# Print briefs for L001, L002, and L006
for letter_id in ["L001", "L002", "L006"]:
    print("=" * 70)
    print(f"BRIEF — {letter_id}")
    print("=" * 70)
    print(briefs[letter_id])
    print("\n")

BRIEF — L001
**Strengths**
- 12 years of experience selling provisions at Makola Market.  
- Consistent monthly profit of GHS 900 reported from the current stall.  
- Saved GHS 2,500 through the institution’s susu scheme over the past two years with a perfect contribution record.  
- Offers a repayment plan of GHS 450 per month for 20 months, which is within the stated profit level.  
- Has a guarantor – her sister, a teacher, who is willing to stand for the loan.  

**Risks / Red Flags**
- Monthly repayment (GHS 450) represents 50 % of the reported monthly profit, leaving limited buffer for unexpected expenses.  
- No formal collateral is mentioned; the guarantee relies solely on a family member.  
- Savings of GHS 2,500 cover only about 31 % of the loan amount, indicating a relatively high loan‑to‑savings ratio.  
- The letter does not provide evidence of the profit figure (e.g., sales records, bank statements).  
- No detail on the cost breakdown of the deep freezer or projected rev

**Student Reasoning — Decision support**

**1. Comparing L003 (strong) vs L006 (weak):**

The system correctly differentiated the two applications, though the given
output shows L001 and L002 rather than L003 directly, comparing L001
(strong, similar profile to L003) against L006 (weak) shows a clear pattern.

For L001, the system correctly identified real strengths grounded in the
letter: 12 years of experience, a documented savings history with a
"perfect contribution record," a realistic repayment-to-profit ratio, and
a named guarantor. Its flagged risks were genuinely proportionate, noting
the repayment amount is a large share of profit, and that the guarantor's
own financial capacity isn't documented, rather than treating the
application as automatically safe just because it has a guarantor.

For L006, the system correctly identified the absence of any real
evidence: no experience, no financial projections, no collateral, and
specifically flagged that launching three unrelated businesses
simultaneously "may stretch limited capital and management capacity",
a genuinely important insight the letter itself doesn't state directly
but is a reasonable risk inference grounded in what was stated (three
ventures, no experience, no capital shown). The system did not get fooled
by Kofi's self-description as "trustworthy" and "business minded", it
correctly noted these were subjective claims ("per friends' feedback")
rather than treating them as evidence.

**2. Why forbid "approve"/"reject":**

Practical reason: the system has no access to bank verification, credit
history, fraud checks, or regulatory requirements that a real lending
decision needs, it only ever sees the text of one letter. An "approve"
output would be a decision made on radically incomplete information,
which could expose the institution to real financial risk.

Ethical reason: automating final loan decisions removes human
accountability and judgment from decisions that materially affect
people's livelihoods. A human loan officer can weigh context, fairness,
and circumstances an LLM cannot, for example, recognizing that someone
who writes poorly in English might still run a solid business. Keeping
a human in the loop preserves the ability to catch cases the automated
system's language-based judgment might unfairly penalize.

Part 3.4 — Commit your prompt templates

In [ ]:
prompts_content = '''"""
Prompt templates for the AfriLingua loan decision-support system.
Lab 4 — Section 3.

Version history:
- SUMMARY: V1 was a bare "Summarize this:" prompt — it produced inconsistent
  formatting (added headers), used interpretive language instead of neutral
  reporting, and did not flag missing information. V2 added a role, explicit
  constraints (3-4 sentences, factual, no invented details), and fixed all
  three issues.
- EXTRACT: built directly as a structured JSON prompt from the start, using
  an explicit schema, a few-shot example NOT drawn from the working dataset,
  and an explicit "use null, do not guess" instruction to prevent hallucinated
  field values.
- BRIEF: combines the raw letter and the extracted JSON, and explicitly
  forbids "approve"/"reject" outputs to keep a human in the loop for the
  final lending decision.
"""

# ── Summarization (Component 1) ────────────────────────────────────────────

SUMMARY_SYSTEM_V2 = """You are an assistant to a microfinance loan officer in Ghana.
Your job is to summarize loan application letters into short, factual briefs
the officer can scan quickly.

Rules:
- Write exactly 3-4 sentences.
- Be strictly factual and neutral in tone — do not add opinions or judgments.
- Do NOT invent, assume, or infer any detail that is not explicitly stated in the letter.
- If a key detail (amount, purpose, repayment plan) is missing or vague, say so plainly
  rather than filling it in.
"""

def SUMMARY_PROMPT_V2(letter_text):
    return f"Summarize this loan application:\\n\\n{letter_text}"


# ── Structured extraction (Component 2) ─────────────────────────────────────

EXTRACT_SYSTEM_PROMPT = """You are a data extraction assistant for a microfinance loan officer.
You will be given a loan application letter. Extract the following fields and
return ONLY a valid JSON object with EXACTLY these keys — no extra text,
no explanation, no markdown formatting:

- applicant_name (string)
- amount_ghs (number)
- purpose (string)
- monthly_profit_ghs (number or null)
- has_collateral_or_guarantor (boolean)
- repayment_months (number or null)

RULES:
- If a field is not explicitly stated in the letter, use null. Do NOT guess or infer.
- Return ONLY the JSON object. No commentary before or after.

EXAMPLE:

Letter:
"Dear Sir, my name is Ama Serwaa. I run a small chop bar in Tema and need
GHS 4,000 to buy new cooking equipment. My guarantor is my brother, a civil
servant. I did not mention my monthly income in this letter."

Output:
{
  "applicant_name": "Ama Serwaa",
  "amount_ghs": 4000,
  "purpose": "buy new cooking equipment for chop bar",
  "monthly_profit_ghs": null,
  "has_collateral_or_guarantor": true,
  "repayment_months": null
}
"""

def EXTRACT_PROMPT(letter_text):
    return f"Extract the fields from this loan application letter:\\n\\n{letter_text}"


# ── Decision-support brief (Component 3) ────────────────────────────────────

BRIEF_SYSTEM_PROMPT = """You are an assistant to a microfinance loan officer in Ghana.
You support the officer's decision-making — you do NOT make the final decision.

Given a loan application letter and its extracted structured data, produce a brief with
exactly these four sections:

1. Strengths — bullet points, grounded strictly in what the letter actually states.
2. Risks / Red Flags — bullet points identifying concerns for the loan officer to consider.
3. Missing Information — what the officer should request before deciding, if anything.
4. Suggested Next Step — ONE of: "invite for interview", "request documents",
   "flag for senior review", or "proceed to standard review".

RULES:
- NEVER output "approve" or "reject" or any equivalent final decision.
- The final lending decision is always made by a human loan officer, not by you.
- Do not invent details not present in the letter or extracted data.
- Be factual and neutral — this is a decision-support tool, not a decision-maker.
"""

def BRIEF_PROMPT(letter_text, extracted_json):
    import json
    return f"""Loan application letter:
{letter_text}

Extracted data:
{json.dumps(extracted_json, indent=2)}

Produce the decision-support brief."""
'''

with open("prompts.py", "w") as f:
    f.write(prompts_content)

print("prompts.py written.")

prompts.py written.


In [ ]:
prompts_content = '''"""
Prompt templates for the AfriLingua loan decision-support system.
Lab 4 — Section 3.

Version history:
- SUMMARY: V1 was a bare "Summarize this:" prompt — it produced inconsistent
  formatting (added headers), used interpretive language instead of neutral
  reporting, and did not flag missing information. V2 added a role, explicit
  constraints (3-4 sentences, factual, no invented details), and fixed all
  three issues.
- EXTRACT: built directly as a structured JSON prompt from the start, using
  an explicit schema, a few-shot example NOT drawn from the working dataset,
  and an explicit "use null, do not guess" instruction to prevent hallucinated
  field values.
- BRIEF: combines the raw letter and the extracted JSON, and explicitly
  forbids "approve"/"reject" outputs to keep a human in the loop for the
  final lending decision.
"""

# ── Summarization (Component 1) ────────────────────────────────────────────

SUMMARY_SYSTEM_V2 = """You are an assistant to a microfinance loan officer in Ghana.
Your job is to summarize loan application letters into short, factual briefs
the officer can scan quickly.

Rules:
- Write exactly 3-4 sentences.
- Be strictly factual and neutral in tone — do not add opinions or judgments.
- Do NOT invent, assume, or infer any detail that is not explicitly stated in the letter.
- If a key detail (amount, purpose, repayment plan) is missing or vague, say so plainly
  rather than filling it in.
"""

def SUMMARY_PROMPT_V2(letter_text):
    return f"Summarize this loan application:\\n\\n{letter_text}"


# ── Structured extraction (Component 2) ─────────────────────────────────────

EXTRACT_SYSTEM_PROMPT = """You are a data extraction assistant for a microfinance loan officer.
You will be given a loan application letter. Extract the following fields and
return ONLY a valid JSON object with EXACTLY these keys — no extra text,
no explanation, no markdown formatting:

- applicant_name (string)
- amount_ghs (number)
- purpose (string)
- monthly_profit_ghs (number or null)
- has_collateral_or_guarantor (boolean)
- repayment_months (number or null)

RULES:
- If a field is not explicitly stated in the letter, use null. Do NOT guess or infer.
- Return ONLY the JSON object. No commentary before or after.

EXAMPLE:

Letter:
"Dear Sir, my name is Ama Serwaa. I run a small chop bar in Tema and need
GHS 4,000 to buy new cooking equipment. My guarantor is my brother, a civil
servant. I did not mention my monthly income in this letter."

Output:
{
  "applicant_name": "Ama Serwaa",
  "amount_ghs": 4000,
  "purpose": "buy new cooking equipment for chop bar",
  "monthly_profit_ghs": null,
  "has_collateral_or_guarantor": true,
  "repayment_months": null
}
"""

def EXTRACT_PROMPT(letter_text):
    return f"Extract the fields from this loan application letter:\\n\\n{letter_text}"


# ── Decision-support brief (Component 3) ────────────────────────────────────

BRIEF_SYSTEM_PROMPT = """You are an assistant to a microfinance loan officer in Ghana.
You support the officer's decision-making — you do NOT make the final decision.

Given a loan application letter and its extracted structured data, produce a brief with
exactly these four sections:

1. Strengths — bullet points, grounded strictly in what the letter actually states.
2. Risks / Red Flags — bullet points identifying concerns for the loan officer to consider.
3. Missing Information — what the officer should request before deciding, if anything.
4. Suggested Next Step — ONE of: "invite for interview", "request documents",
   "flag for senior review", or "proceed to standard review".

RULES:
- NEVER output "approve" or "reject" or any equivalent final decision.
- The final lending decision is always made by a human loan officer, not by you.
- Do not invent details not present in the letter or extracted data.
- Be factual and neutral — this is a decision-support tool, not a decision-maker.
"""

def BRIEF_PROMPT(letter_text, extracted_json):
    import json
    return f"""Loan application letter:
{letter_text}

Extracted data:
{json.dumps(extracted_json, indent=2)}

Produce the decision-support brief."""
'''

with open("prompts.py", "w") as f:
    f.write(prompts_content)

print("prompts.py written.")

prompts.py written.


In [ ]:
prompts_content = '''"""
Prompt templates for the AfriLingua loan decision-support system.
Lab 4 — Section 3.

Version history:
- SUMMARY: V1 was a bare "Summarize this:" prompt — it produced inconsistent
  formatting (added headers), used interpretive language instead of neutral
  reporting, and did not flag missing information. V2 added a role, explicit
  constraints (3-4 sentences, factual, no invented details), and fixed all
  three issues.
- EXTRACT: built directly as a structured JSON prompt from the start, using
  an explicit schema, a few-shot example NOT drawn from the working dataset,
  and an explicit "use null, do not guess" instruction to prevent hallucinated
  field values.
- BRIEF: combines the raw letter and the extracted JSON, and explicitly
  forbids "approve"/"reject" outputs to keep a human in the loop for the
  final lending decision.
"""

# ── Summarization (Component 1) ────────────────────────────────────────────

SUMMARY_SYSTEM_V2 = """You are an assistant to a microfinance loan officer in Ghana.
Your job is to summarize loan application letters into short, factual briefs
the officer can scan quickly.

Rules:
- Write exactly 3-4 sentences.
- Be strictly factual and neutral in tone — do not add opinions or judgments.
- Do NOT invent, assume, or infer any detail that is not explicitly stated in the letter.
- If a key detail (amount, purpose, repayment plan) is missing or vague, say so plainly
  rather than filling it in.
"""

def SUMMARY_PROMPT_V2(letter_text):
    return f"Summarize this loan application:\\n\\n{letter_text}"


# ── Structured extraction (Component 2) ─────────────────────────────────────

EXTRACT_SYSTEM_PROMPT = """You are a data extraction assistant for a microfinance loan officer.
You will be given a loan application letter. Extract the following fields and
return ONLY a valid JSON object with EXACTLY these keys — no extra text,
no explanation, no markdown formatting:

- applicant_name (string)
- amount_ghs (number)
- purpose (string)
- monthly_profit_ghs (number or null)
- has_collateral_or_guarantor (boolean)
- repayment_months (number or null)

RULES:
- If a field is not explicitly stated in the letter, use null. Do NOT guess or infer.
- Return ONLY the JSON object. No commentary before or after.

EXAMPLE:

Letter:
"Dear Sir, my name is Ama Serwaa. I run a small chop bar in Tema and need
GHS 4,000 to buy new cooking equipment. My guarantor is my brother, a civil
servant. I did not mention my monthly income in this letter."

Output:
{
  "applicant_name": "Ama Serwaa",
  "amount_ghs": 4000,
  "purpose": "buy new cooking equipment for chop bar",
  "monthly_profit_ghs": null,
  "has_collateral_or_guarantor": true,
  "repayment_months": null
}
"""

def EXTRACT_PROMPT(letter_text):
    return f"Extract the fields from this loan application letter:\\n\\n{letter_text}"


# ── Decision-support brief (Component 3) ────────────────────────────────────

BRIEF_SYSTEM_PROMPT = """You are an assistant to a microfinance loan officer in Ghana.
You support the officer's decision-making — you do NOT make the final decision.

Given a loan application letter and its extracted structured data, produce a brief with
exactly these four sections:

1. Strengths — bullet points, grounded strictly in what the letter actually states.
2. Risks / Red Flags — bullet points identifying concerns for the loan officer to consider.
3. Missing Information — what the officer should request before deciding, if anything.
4. Suggested Next Step — ONE of: "invite for interview", "request documents",
   "flag for senior review", or "proceed to standard review".

RULES:
- NEVER output "approve" or "reject" or any equivalent final decision.
- The final lending decision is always made by a human loan officer, not by you.
- Do not invent details not present in the letter or extracted data.
- Be factual and neutral — this is a decision-support tool, not a decision-maker.
"""

def BRIEF_PROMPT(letter_text, extracted_json):
    import json
    return f"""Loan application letter:
{letter_text}

Extracted data:
{json.dumps(extracted_json, indent=2)}

Produce the decision-support brief."""
'''

with open("prompts.py", "w") as f:
    f.write(prompts_content)

print("prompts.py written.")

prompts.py written.


In [ ]:
!git add prompts.py
!git commit -m "Add prompts.py: SUMMARY_PROMPT_V2, EXTRACT_PROMPT, BRIEF_PROMPT with version history in docstring"
!git push

Author identity unknown

*** Please tell me who you are.

Run

  git config --global user.email "you@example.com"
  git config --global user.name "Your Name"

to set your account's default identity.
Omit --global to set the identity only in this repository.

fatal: unable to auto-detect email address (got 'root@45ed40713897.(none)')
fatal: could not read Username for 'https://github.com': No such device or address


In [ ]:
!git log -1 --format="%H"

c01b96f8c64e5d798a64990376ef0c40aee1681d


> **Commit hash:**c01b96f8c64e5d798a64990376ef0c40aee1681d




Section 4 — Evaluation: Quality, Reliability, Appropriateness

Part 4.1 — Extraction accuracy against gold labels

In [ ]:
# Part 4.1: Extraction accuracy against gold labels

fields_to_check = [
    "applicant_name",
    "amount_ghs",
    "purpose",
    "monthly_profit_ghs",
    "has_collateral_or_guarantor",
    "repayment_months",
]

gold_letter_ids = list(GOLD.keys())  # L001, L003, L006

accuracy_table = {}

for field in fields_to_check:
    row = {}
    correct_count = 0

    for letter_id in gold_letter_ids:
        gold_value = GOLD[letter_id][field]
        extracted_value = (
            extraction_results[letter_id].get(field)
            if extraction_results[letter_id]
            else None
        )

        if field == "applicant_name":
            # Case-insensitive match for names
            is_match = (
                str(gold_value).strip().lower() == str(extracted_value).strip().lower()
                if extracted_value is not None
                else False
            )
        elif field == "purpose":
            # Purpose is free text — use a loose substring/keyword check rather than exact match
            is_match = extracted_value is not None and any(
                word.lower() in str(extracted_value).lower()
                for word in str(gold_value).lower().split()
                if len(word) > 4
            )
        else:
            # Exact match for numbers and booleans (including None == None)
            is_match = gold_value == extracted_value

        row[letter_id] = (
            "✅" if is_match else f"❌ (gold={gold_value}, got={extracted_value})"
        )
        if is_match:
            correct_count += 1

    row["accuracy"] = f"{correct_count}/{len(gold_letter_ids)}"
    accuracy_table[field] = row

accuracy_df = pd.DataFrame(accuracy_table).T
print("=== Extraction Accuracy vs Gold Labels ===")
display(accuracy_df)

overall_correct = sum(
    1
    for field in fields_to_check
    for letter_id in gold_letter_ids
    if (
        GOLD[letter_id][field]
        == (
            extraction_results[letter_id].get(field)
            if extraction_results[letter_id]
            else None
        )
    )
    or (
        field == "applicant_name"
        and str(GOLD[letter_id][field]).lower()
        == str(extraction_results[letter_id].get(field, "")).lower()
    )
)
total_checks = len(fields_to_check) * len(gold_letter_ids)
print(f"\nOverall accuracy: {overall_correct}/{total_checks}")

=== Extraction Accuracy vs Gold Labels ===


,L001,L003,L006,accuracy
applicant_name,✅,✅,✅,3/3
amount_ghs,✅,✅,✅,3/3
purpose,✅,✅,✅,3/3
monthly_profit_ghs,✅,✅,✅,3/3
has_collateral_or_guarantor,✅,✅,✅,3/3
repayment_months,✅,✅,✅,3/3



Overall accuracy: 15/18


Part 4.2 — Reliability: is the system consistent?

In [ ]:
# Part 4.2: Reliability — is the system consistent?

l004_text = LETTERS["L004"]

# 5 runs at temperature = 0
print("=== Reliability Test — L004 at temperature=0 ===\n")
results_temp0 = []
for i in range(5):
    result = extract_fields(
        l004_text
    )  # Note: extract_fields uses temperature=0 internally
    results_temp0.append(result)
    print(f"Run {i + 1}: {result}")

# 5 runs at temperature = 1.0
# extract_fields is hardcoded to temperature=0, so we call the LLM directly here
print("\n=== Reliability Test — L004 at temperature=1.0 ===\n")
results_temp1 = []
for i in range(5):
    raw_output = ask_llm(
        EXTRACT_PROMPT(l004_text),
        system_prompt=EXTRACT_SYSTEM_PROMPT,
        temperature=1.0,
        max_tokens=800,
    )
    cleaned = re.sub(
        r"^```json\s*|\s*```$", "", raw_output.strip(), flags=re.MULTILINE
    ).strip()
    try:
        result = json.loads(cleaned)
    except json.JSONDecodeError:
        result = None
        print(f"⚠️ Run {i + 1}: failed to parse JSON")
    results_temp1.append(result)
    print(f"Run {i + 1}: {result}")


# Analyze consistency
def analyze_consistency(results, label):
    valid_json_count = sum(1 for r in results if r is not None)
    json_strings = [json.dumps(r, sort_keys=True) for r in results if r is not None]
    unique_count = len(set(json_strings))

    print(f"\n{label}:")
    print(f"  Valid JSON: {valid_json_count}/5")
    print(
        f"  Unique outputs: {unique_count} (1 = perfectly consistent, 5 = all different)"
    )


analyze_consistency(results_temp0, "Temperature = 0")
analyze_consistency(results_temp1, "Temperature = 1.0")

=== Reliability Test — L004 at temperature=0 ===

Run 1: {'applicant_name': 'Yaw Owusu', 'amount_ghs': 12000, 'purpose': 'feed and 500 new layers for poultry farm', 'monthly_profit_ghs': 1500, 'has_collateral_or_guarantor': True, 'repayment_months': 18}
Run 2: {'applicant_name': 'Yaw Owusu', 'amount_ghs': 12000, 'purpose': 'feed and new layers for poultry farm', 'monthly_profit_ghs': 1500, 'has_collateral_or_guarantor': True, 'repayment_months': 18}
Run 3: {'applicant_name': 'Yaw Owusu', 'amount_ghs': 12000, 'purpose': 'feed and new layers for poultry farm', 'monthly_profit_ghs': 1500, 'has_collateral_or_guarantor': True, 'repayment_months': 18}
Run 4: {'applicant_name': 'Yaw Owusu', 'amount_ghs': 12000, 'purpose': 'feed and new layers for poultry farm', 'monthly_profit_ghs': 1500, 'has_collateral_or_guarantor': True, 'repayment_months': 18}
Run 5: {'applicant_name': 'Yaw Owusu', 'amount_ghs': 12000, 'purpose': 'feed and 500 new layers for poultry farm', 'monthly_profit_ghs': 1500, 'ha

Part 4.3 — Hallucination probing

In [ ]:
# Part 4.3: Hallucination probing

# Test 1: Ask about a detail NOT present in a letter
print("=== Test 1: Asking about absent information (L001) ===\n")

test1_question = f"""Based on this loan application letter, what is the applicant's credit score?

Letter:
{LETTERS["L001"]}"""

test1_answer = ask_llm(
    test1_question,
    system_prompt="You are an assistant to a microfinance loan officer. Only use information explicitly stated in the letter. If information is not present, say so clearly.",
    temperature=0,
)

print(f"Response:\n{test1_answer}\n")

# Manually inspect: does it admit absence, or invent a number?
test1_verdict = (
    "PASS"
    if any(
        phrase in test1_answer.lower()
        for phrase in [
            "not mention",
            "not state",
            "not provided",
            "no credit score",
            "does not",
            "not include",
            "not specify",
        ]
    )
    else "FAIL"
)
print(f"Verdict: {test1_verdict}")


# Test 2: Feed an empty/irrelevant text into the extractor
print("\n\n=== Test 2: Extracting from irrelevant text (weather report) ===\n")

irrelevant_text = """Weather Report for Accra — Tuesday, 20 August 2026

Today will be mostly cloudy with a high of 29°C and a low of 24°C.
Humidity will remain elevated at around 85% throughout the day, with
a light breeze from the southwest at 10-15 km/h. There is a 40% chance
of afternoon showers. Tomorrow's forecast shows similar conditions
continuing into the weekend."""

test2_result = extract_fields(irrelevant_text)

print(f"Extraction result:\n{test2_result}\n")

# Verdict: did it correctly return nulls/None, or did it fabricate an applicant?
if test2_result is None:
    test2_verdict = "PASS (failed to parse — did not fabricate structured data)"
elif (
    test2_result.get("applicant_name") is None
    and test2_result.get("amount_ghs") is None
):
    test2_verdict = "PASS (correctly returned null fields, no fabrication)"
else:
    test2_verdict = f"FAIL (fabricated data: {test2_result})"

print(f"Verdict: {test2_verdict}")

=== Test 1: Asking about absent information (L001) ===

Response:
The loan application letter does not provide any information about the applicant’s credit score.

Verdict: PASS


=== Test 2: Extracting from irrelevant text (weather report) ===

Extraction result:
{'applicant_name': None, 'amount_ghs': None, 'purpose': None, 'monthly_profit_ghs': None, 'has_collateral_or_guarantor': None, 'repayment_months': None}

Verdict: PASS (correctly returned null fields, no fabrication)


**Student Reasoning — Evaluation results**

**1. Extraction accuracy:**
The system achieved 18/18 (100%) accuracy across all 6 fields on the 3
gold-labeled letters (L001, L003, L006), every field matched the gold
standard exactly, including correctly returning null where information
was genuinely absent (e.g. L006's monthly_profit_ghs). No single field
was hardest based on this test set, though the purpose field required
the most careful comparison since it's free text, the model paraphrases
the letter's wording rather than quoting it verbatim, so scoring it
required a substring check rather than an exact match.

**2. What the reliability experiment showed:**
At temperature=0, the extraction was NOT perfectly identical across
5 runs, it produced 2 unique outputs, varying only in whether "500"
was included in the purpose field wording. At temperature=1.0, it was
less consistent, producing 4 unique outputs out of 5, again varying
mainly in the phrasing of the purpose field, not the structured numeric
fields (amount, profit, months all stayed identical every time). This
shows that even at temperature=0, LLMs are not perfectly deterministic,
minor wording variation in free-text fields can still occur. For a
production system, this means the numeric/boolean fields can be trusted
as reliable even at temp=0, but any free-text field should either be
post-processed/normalized or not relied upon for exact string matching
in downstream systems.

**3. Did the system hallucinate under probing?**
No, both adversarial tests passed. Test 1 correctly stated the letter
"does not mention a credit score" rather than inventing one. Test 2,
given an irrelevant weather report, correctly returned null for every
single field rather than fabricating a fake applicant. This confirms
the "use null, do not guess" instruction and the explicit system role
constraints are effective at preventing hallucination in these two
tested scenarios. To further reduce risk in production, the system
could add a confidence check or a secondary validation pass that flags
extractions where the source text doesn't contain any of the expected
loan-application keywords (amount, name, purpose) before trusting the
output at all.

Part 4.4 — Appropriateness: should this system exist?

**Student Reasoning — Appropriateness**

**1. Who could be unfairly harmed by full automation?**

The system's summarization and extraction steps rely heavily on how clearly
and fluently the letter is written. L002 (Kwame Boateng) is a real example
from our own dataset, his letter is short, uses vague phrases like "God
willing everything will be fine" and "whenever the money comes," and was
flagged with multiple red flags by our system. But nothing in the letter
tells us whether his trotro business is actually profitable, he simply
didn't write it down clearly. A more articulate applicant with the exact
same underlying business could describe the same situation in a way that
scores much better, purely because of writing skill, not creditworthiness.

This creates a real risk: applicants who speak English as a second or
third language, who have less formal education, or who are less practiced
at writing persuasive letters could be unfairly declined even if their
business is genuinely solid, while applicants who write well (regardless
of business quality) get a more favorable brief. Our own system explicitly
forbids "approve/reject" for exactly this reason, but if a bank chose to
fully automate anyway, this bias would transfer directly into lending
outcomes, disproportionately harming applicants who are disadvantaged in
formal English writing, not in business viability.

**2. Data privacy implications of a third-party, foreign API:**

Loan letters contain a person's name, income, business details, and
financial history, this is sensitive personal and financial data. Sending
it to Groq's API means this data leaves Ghana and is processed on servers
in another jurisdiction, subject to that country's data protection laws
rather than Ghana's. Before deploying this at a real microfinance
institution, I would check: (1) whether Groq's terms of service and data
retention policy allow processing of financial personal data and whether
they store or use it for any other purpose (e.g. model training);
(2) whether this complies with Ghana's Data Protection Act (2012) and
any Bank of Ghana / microfinance regulatory requirements around where
customer financial data may be processed and stored; (3) whether
applicants have given informed consent for their letter to be processed
by a third-party AI system at all.

**3. Two concrete production safeguards:**

First, a mandatory human review gate: no brief generated by this system
should ever be visible to the applicant or trigger any action (approve,
decline, request documents) without a human loan officer first reading
both the original letter and the brief, and explicitly signing off. This
directly enforces the "human in the loop" principle the prompts already
encode, but makes it a system-level requirement, not just a prompt
instruction that could be bypassed.

Second, full audit logging with an appeal path: every extraction and
brief generated should be logged alongside the original letter, a
timestamp, and which model/prompt version produced it. If an applicant
is declined, they should have a right to request a human re-review of
their original letter, independent of the AI-generated brief, this
gives a corrective path for exactly the kind of writing-style bias
identified in question 1.


Section 5 — Reflection

**Section 5 - Reflection**

**1. Prompting as engineering, similar to and different from Lab 3 hyperparameters:**
Both are iterative and empirical, you can't reason your way to the right prompt
or the right learning rate purely on paper, you have to run it, look at the
result, and adjust. Just like lr=1e-4/1e-3/1e-1 in Lab 3 gave three completely
different training outcomes, SUMMARY_PROMPT V1 vs V2 gave measurably different
output quality. The key difference is what you're tuning: hyperparameters
adjust numbers that control an already-fixed model architecture, while a
prompt is closer to writing actual instructions in natural language, the
"parameter space" is language itself, which is far less structured and much
harder to search systematically. You can't grid-search prompts the way you
can grid-search learning rates.

**2. Trust, would I run this unattended?**
No. The single result that most influenced this is the reliability experiment
in Part 4.2: even at temperature=0, the extraction was not perfectly
consistent across 5 identical runs (2 unique outputs out of 5). If the exact
same letter can produce two different outputs, that's already enough
uncertainty to require a human check, especially for a task where the output
feeds into a real financial decision. The 18/18 accuracy score is reassuring,
but it was measured on only 3 letters, nowhere near enough to trust blindly
at production scale, and the reliability gap shows the system can drift even
without any change in input.

**3. Cost and scale:**
Based on observed usage, a full three-call pipeline (summarize + extract +
brief) per application costs roughly 2,000-2,500 tokens: summarization
averaged about 460 tokens per letter, extraction (with its long schema and
few-shot example in the system prompt) is estimated around 700-900 tokens,
and the brief (longest output, 4 structured sections) is estimated around
900-1,000 tokens. At roughly 2,200 tokens per application, processing 1,000
applications/month would need approximately 2.2 million tokens/month. This
is a moderate volume, well within Groq's free tier for a pilot, but at
real production scale (especially combined with reliability requiring
possible re-runs), this pushes toward needing a paid tier or a
cost-per-token comparison across providers before committing.

**4. Why an API beats training your own model here, and when it wouldn't:**
For this task, calling an API wins because the underlying capability needed, fluent, general-purpose language understanding across arbitrary free-text
letters, would require an enormous amount of data and compute to train
from scratch, far beyond what a microfinance institution could justify for
one internal tool. The Lab 3 neural network took 50 epochs and still only
reached 95% on a narrow, well-defined 7-class problem with clean structured
data; loan letters are open-ended natural language with no fixed structure.
Training a custom model competitive with a foundation model on this kind
of task is simply not realistic at this scale.

An API would NOT be the right choice if: the data is too sensitive to send
to a third party at all (as discussed in Part 4.4), if the task needs to
run fully offline or with guaranteed low latency, if per-call costs at true
scale (millions of applications) would exceed the cost of training and
hosting a smaller specialized model, or if the task is narrow and
well-defined enough that a small trained classifier (like our Lab 2 Random
Forest, which beat our own Lab 3 neural network on both accuracy and speed)
would genuinely outperform a general-purpose LLM anyway.